# NeuroGolf 400 Tasks Auto-Trainer
Trains tiny CNNs on all 400 tasks. Exports to ONNX and zips for submission.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
import os
import sys
import zipfile
import warnings
warnings.filterwarnings('ignore')

KAGGLE_INPUT = '/kaggle/input/neurogolf-2026'
if not os.path.exists(KAGGLE_INPUT):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'task001.json' in files:
            KAGGLE_INPUT = root
            break

print(f"Using dataset path: {KAGGLE_INPUT}")

def convert_to_numpy(example):
    benchmark = {}
    example_shape = (1, 10, 30, 30)
    for mode in ["input", "output"]:
        benchmark[mode] = np.zeros(example_shape, dtype=np.float32)
        grid = example[mode]
        if max(len(grid), len(grid[0])) > 30: return None
        for r, row in enumerate(grid):
            for c, color in enumerate(row):
                benchmark[mode][0][color][r][c] = 1.0
    return benchmark

class Arch1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Conv2d(10, 10, kernel_size=1, bias=False)
    def forward(self, x): return self.net(x)

class Arch2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Conv2d(10, 10, kernel_size=3, padding=1)
    def forward(self, x): return self.net(x)

class Arch3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(10, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 10, kernel_size=3, padding=1)
        )
    def forward(self, x): return self.net(x)

class IdentityModel(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x): return x

os.makedirs('submission', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

solved_count = 0

for task_id in range(1, 401):
    file_path = os.path.join(KAGGLE_INPUT, f"task{task_id:03d}.json")
    if not os.path.exists(file_path): 
        print(f"Missing {file_path}")
        continue
        
    with open(file_path, "r") as f:
        task_data = json.load(f)
        
    all_examples = task_data.get("train", []) + task_data.get("test", []) + task_data.get("arc-gen", [])
    
    X_train, Y_train = [], []
    valid = True
    for ex in all_examples:
        bench = convert_to_numpy(ex)
        if not bench:
            valid = False
            break
        X_train.append(bench["input"])
        Y_train.append(np.argmax(bench["output"].squeeze(0), axis=0))
        
    if not valid or not X_train:
        # Use Identity fallback
        model = IdentityModel().eval()
        torch.onnx.export(model, torch.randn(1, 10, 30, 30), f"submission/task{task_id:03d}.onnx", 
                          input_names=['input'], output_names=['output'])
        continue
        
    X_tensor = torch.tensor(np.array(X_train)).squeeze(1).to(device)
    Y_tensor = torch.tensor(np.array(Y_train)).long().to(device)
    
    best_model = None
    is_solved = False
    
    for Arch in [Arch1, Arch2, Arch3]:
        model = Arch().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        
        for epoch in range(300):
            optimizer.zero_grad()
            outputs = model(X_tensor)
            loss = criterion(outputs, Y_tensor)
            loss.backward()
            optimizer.step()
            
            if loss.item() < 0.05:
                preds = torch.argmax(outputs, dim=1)
                if (preds == Y_tensor).all().item():
                    best_model = model
                    is_solved = True
                    break
        if is_solved:
            break
            
    if is_solved:
        solved_count += 1
        print(f"Task {task_id:03d} SOLVED with {best_model.__class__.__name__}")
        best_model.eval().cpu()
        torch.onnx.export(best_model, torch.randn(1, 10, 30, 30), f"submission/task{task_id:03d}.onnx",
                          input_names=['input'], output_names=['output'])
    else:
        # Fallback
        model = IdentityModel().eval()
        torch.onnx.export(model, torch.randn(1, 10, 30, 30), f"submission/task{task_id:03d}.onnx",
                          input_names=['input'], output_names=['output'])

print(f"\nTotal tasks perfectly solved: {solved_count} / 400")

# Zip submission
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for i in range(1, 401):
        filename = f"task{i:03d}.onnx"
        filepath = os.path.join("submission", filename)
        if os.path.exists(filepath):
            zf.write(filepath, filename)

print("Created submission.zip successfully!")
